# DP Training Results: Full-Backbone ConvNeXtTiny Scatter Plot

Interactive Plotly figure: target epsilon (eps 2–10) and noise multiplier mode DP training results,
with non-DP training marked as baseline. Hover over points for details.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [ ]:
NOTEBOOK_DIR = Path.cwd()
CSV_PATH = NOTEBOOK_DIR.parents[0] / "results_summary.csv"
ARCHIVES_ROOT = NOTEBOOK_DIR.parents[1] / "centralized_runs" / "archive"
FIGURES_DIR = NOTEBOOK_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} runs")

In [ ]:
def build_noise_lookup():
    """Scan archived metrics.json files for target-epsilon noise_multiplier values."""
    lookup = {}
    for metrics_path in ARCHIVES_ROOT.rglob("**/metrics.json"):
        try:
            with open(metrics_path) as f:
                data = json.load(f)
        except (json.JSONDecodeError, OSError):
            continue
        dp_info = data.get("differential_privacy", {})
        if dp_info.get("mode") == "target_epsilon":
            nm = dp_info.get("noise_multiplier")
            if nm is not None:
                lookup[metrics_path.parent.name] = float(nm)
    return lookup

noise_lookup = build_noise_lookup()
print(f"Noise multiplier lookup: {len(noise_lookup)} entries")

In [ ]:
fb = df[
    (df["FreezeBackbone"] == False)
    & (df["Status"] == "completed")
].copy()

for col in ["BestValAUC", "FinalEpsilon", "TargetEpsilon", "NoiseMultiplier",
            "BestValBalancedAccuracy", "FinalTestAUC", "FinalTestBalancedAccuracy",
            "BestValSensitivity", "BestValSpecificity",
            "FinalTestSensitivity", "FinalTestSpecificity"]:
    fb[col] = pd.to_numeric(fb[col], errors="coerce")

print(f"Full-backbone, completed: {len(fb)} runs")

## Baseline (non-DP)

In [ ]:
baseline = fb[(fb["DPEnabled"] == False) & (fb["BestValAUC"] > 0.5)]
baseline_auc = baseline["BestValAUC"].max()
baseline_run = baseline.loc[baseline["BestValAUC"].idxmax()]
print(f"Baseline: {baseline_run['RunName']}")
print(f"  BestValAUC: {baseline_auc:.4f}")

## Target epsilon mode (eps 2–10)

In [ ]:
target_eps = fb[
    (fb["DPMode"] == "target_epsilon")
    & (fb["TargetEpsilon"] >= 2)
    & (fb["TargetEpsilon"] <= 10)
    & (fb["BestValAUC"] > 0.5)
].copy()
target_eps["Sigma"] = target_eps["RunName"].map(noise_lookup)

cols = ["TargetEpsilon", "FinalEpsilon", "MaxGradNorm", "Epochs", "Sigma", "BestValAUC", "FinalTestAUC"]
display(target_eps[cols].rename(columns={"Sigma": "σ"}).set_index(np.arange(len(target_eps)) + 1))

## Noise multiplier mode

In [ ]:
noise_mode = fb[
    (fb["DPMode"] == "noise_multiplier")
    & (fb["NoiseMultiplier"] > 0)
    & (fb["FinalEpsilon"] > 0)
    & (fb["FinalEpsilon"] <= 200)
    & (fb["BestValAUC"] > 0.5)
    & (fb["ArchivePath"].str.contains("fullbackbone_dp_diagnostics|fullbackbone_dp_targeteps", na=False))
].copy()
noise_mode = noise_mode.drop_duplicates(subset=["NoiseMultiplier", "MaxGradNorm", "Epochs"])
noise_mode["Sigma"] = noise_mode["NoiseMultiplier"]

cols = ["Sigma", "FinalEpsilon", "MaxGradNorm", "Epochs", "BestValAUC", "FinalTestAUC"]
display(noise_mode[cols].rename(columns={"Sigma": "σ"}).set_index(np.arange(len(noise_mode)) + 1))

## Interactive Scatter Plot

In [ ]:
def make_hover(row, mode_label):
    parts = [
        f"<b>{row['RunName']}</b>",
        f"Mode: {mode_label}",
        f"σ: {row['Sigma']:.4f}" if pd.notna(row.get("Sigma")) else "σ: N/A",
        f"Target ε: {row['TargetEpsilon']}" if pd.notna(row.get("TargetEpsilon")) else "",
        f"Final ε: {row['FinalEpsilon']:.2f}",
        f"Clip: {row['MaxGradNorm']}",
        f"Epochs: {row['Epochs']}",
        f"Best Val AUC: {row['BestValAUC']:.4f}",
        f"Best Val BalAcc: {row['BestValBalancedAccuracy']:.4f}",
        f"Best Val Sens: {row['BestValSensitivity']:.4f}",
        f"Best Val Spec: {row['BestValSpecificity']:.4f}",
        f"Final Test AUC: {row['FinalTestAUC']:.4f}",
        f"Final Test BalAcc: {row['FinalTestBalancedAccuracy']:.4f}",
        f"Final Test Sens: {row['FinalTestSensitivity']:.4f}",
        f"Final Test Spec: {row['FinalTestSpecificity']:.4f}",
    ]
    return "<br>".join(p for p in parts if p)


fig = go.Figure()

fig.add_hline(
    y=baseline_auc,
    line_dash="dash",
    line_color="green",
    annotation_text=f"Non-DP baseline (AUC={baseline_auc:.4f})",
    annotation_position="right",
)

non_dp_hover = (
    f"<b>{baseline_run['RunName']}</b><br>"
    f"Mode: non-DP (no privacy)<br>"
    f"Epochs: {baseline_run['Epochs']}<br>"
    f"Optimizer: {baseline_run['Optimizer']}<br>"
    f"LrBackbone: {baseline_run['LrBackbone']}<br>"
    f"LrFinetune: {baseline_run['LrFinetune']}<br>"
    f"BatchSize: {baseline_run['BatchSize']}<br>"
    f"Best Val AUC: {baseline_run['BestValAUC']:.4f}<br>"
    f"Best Val BalAcc: {baseline_run['BestValBalancedAccuracy']:.4f}<br>"
    f"Best Val Sens: {baseline_run['BestValSensitivity']:.4f}<br>"
    f"Best Val Spec: {baseline_run['BestValSpecificity']:.4f}<br>"
    f"Final Test AUC: {baseline_run['FinalTestAUC']:.4f}<br>"
    f"Final Test BalAcc: {baseline_run['FinalTestBalancedAccuracy']:.4f}<br>"
    f"Final Test Sens: {baseline_run['FinalTestSensitivity']:.4f}<br>"
    f"Final Test Spec: {baseline_run['FinalTestSpecificity']:.4f}"
)

fig.add_trace(go.Scatter(
    x=[29.5],
    y=[baseline_auc],
    mode="markers+text",
    name="Non-DP baseline",
    marker=dict(size=16, color="green", symbol="star", line=dict(width=1.5, color="black")),
    text=["★ baseline"],
    textposition="top center",
    textfont=dict(size=9, color="green"),
    hovertemplate="%{hovertext}<extra></extra>",
    hovertext=[non_dp_hover],
))

fig.add_trace(go.Scatter(
    x=target_eps["FinalEpsilon"],
    y=target_eps["BestValAUC"],
    mode="markers+text",
    name=f"Target epsilon mode (n={len(target_eps)})",
    marker=dict(size=10, color="#2166ac", line=dict(width=1, color="black")),
    text=[f"σ={s:.3f}" for s in target_eps["Sigma"]],
    textposition="bottom center",
    textfont=dict(size=7, color="#2166ac"),
    hovertemplate="%{hovertext}<extra></extra>",
    hovertext=[make_hover(row, "target_epsilon") for _, row in target_eps.iterrows()],
))

fig.add_trace(go.Scatter(
    x=noise_mode["FinalEpsilon"],
    y=noise_mode["BestValAUC"],
    mode="markers+text",
    name=f"Noise multiplier mode (n={len(noise_mode)})",
    marker=dict(size=12, color="#d6604d", symbol="triangle-up", line=dict(width=1, color="black")),
    text=[f"σ={s}" for s in noise_mode["Sigma"]],
    textposition="top center",
    textfont=dict(size=8, color="#d6604d"),
    hovertemplate="%{hovertext}<extra></extra>",
    hovertext=[make_hover(row, "noise_multiplier") for _, row in noise_mode.iterrows()],
))

fig.update_layout(
    title="DP Training Results: Full-Backbone ConvNeXtTiny",
    xaxis_title="Final Epsilon",
    yaxis_title="Best Validation AUC",
    xaxis_range=[0, 32],
    yaxis_range=[0.55, max(
        target_eps["BestValAUC"].max(),
        noise_mode["BestValAUC"].max(),
        baseline_auc,
    ) * 1.03],
    legend=dict(x=0.99, y=0.01, xanchor="right", yanchor="bottom"),
    hoverlabel=dict(font_size=12),
)

fig.write_html(FIGURES_DIR / "dp_scatter_target_eps.html")
fig.show()